# 풀스택 GPT: #6.0 ~ #6.10

- [x] Implement a complete RAG pipeline with a Stuff Documents chain.
- [x] You must implement the chain manually.
- [x] Give a ConversationBufferMemory to the chain.
- [x] Use this document to perform RAG: https://gist.github.com/serranoarevalo/5acf755c2b8d83f1707ef266b82ea223
- Ask the following questions to the chain:
    - [x] Is Aaronson guilty?
    - [x] What message did he write in the table?
    - [x] Who is Julia?

In [7]:
from langchain.memory import ConversationBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, CacheBackedEmbeddings
from langchain.storage import LocalFileStore
from langchain.vectorstores.faiss import FAISS
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

def load_memory(_):
    return memory.load_memory_variables({})["history"]

def invoke_chain(question):
    result = chain.invoke(question)
    memory.save_context({"input": question}, {"output": result.content})

cache_dir = LocalFileStore("../.cache/")
memory = ConversationBufferMemory(return_messages=True)
llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    temperature=0.1,
)

loader = TextLoader("../files/document.txt")
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n\n",
    chunk_size=500,
    chunk_overlap=50,
)
docs = loader.load_and_split(splitter)

embeddings = OpenAIEmbeddings()
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)
vectorstore = FAISS.from_documents(docs, cached_embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question using only the following context and chat history. If you don't know the answer, just say you don't know. Don't make it up:\n\n{context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

chain = {"context": retriever, "history": RunnableLambda(load_memory), "question": RunnablePassthrough()} | prompt | llm

Created a chunk of size 538, which is longer than the specified 500
Created a chunk of size 511, which is longer than the specified 500


In [9]:
questions = [
    "Is Aaronson guilty?", 
    "What message did he write in the table?", 
    "Who is Julia?",
]

for question in questions:
    result = chain.invoke(question)
    print(f"Q: {question}\nA: {result}\n")

According to the context provided, Winston believes that Jones, Aaronson, and Rutherford are guilty of the crimes they are charged with, and he has convinced himself that he has never seen the photograph that disproved their guilt. Therefore, he accepts that Aaronson is guilty.q: Is Aaronson guilty?
a: content='According to the context provided, Winston believes that Jones, Aaronson, and Rutherford are guilty of the crimes they are charged with, and he has convinced himself that he has never seen the photograph that disproved their guilt. Therefore, he accepts that Aaronson is guilty.'
He wrote "FREEDOM IS SLAVERY" and beneath it "TWO AND TWO MAKE FIVE" and then "GOD IS POWER."q: What message did he write in the table?
a: content='He wrote "FREEDOM IS SLAVERY" and beneath it "TWO AND TWO MAKE FIVE" and then "GOD IS POWER."'
Julia is a character in the context provided, who is loved by Winston. She is someone with whom Winston shares a deep emotional connection, and he expresses a stron